# 第14回 架空のAI二方式を評価する

目標：条件・分母・限界を添えて結果を説明する。実AIの学習・生成・実測は行いません。

既存venvのカーネルを選び、再起動して全セル実行してください。同梱Pythonのimportは不要。基準入力・結果を保存し、コピー上で一項目ずつ変更します。

In [ ]:
"""第14回: 架空の固定結果を比較する。AIの学習・生成・実測ではない。"""
from copy import deepcopy
import argparse
import json

DATA = {
    'A': {'normal': {'correct': [1,1,1,1,1,0], 'scores': [95,90,85,80,75,90]},
          'changed': {'correct': [1,1,0,0,0,0], 'scores': [95,80,75,65,55,45]}},
    'B': {'normal': {'correct': [1,1,1,1,0,0], 'scores': [90,85,80,75,70,60]},
          'changed': {'correct': [1,1,1,1,0,0], 'scores': [85,80,75,70,65,55]}},
}
BASE = dict(threshold=0, normal_percent=50, review_cost=4, cost_a=1,
            cost_b=3, min_accuracy=80, min_coverage=50)


def integer(value, name):
    if type(value) is not int or not 0 <= value <= 100:
        raise ValueError(f'{name}: 0〜100の整数を指定してください')
    return value


def configuration(config=None):
    if config is None:
        return deepcopy(BASE)
    if not isinstance(config, dict) or set(config) - set(BASE):
        raise ValueError('未知の設定です')
    result = dict(BASE, **config)
    for key, value in result.items():
        integer(value, key)
    return result


def records(method, group='all'):
    if method not in ('A', 'B') or group not in ('all', 'normal', 'changed'):
        raise ValueError('未知の方式または条件です')
    rows = []
    for kind in ('normal', 'changed'):
        if group not in ('all', kind):
            continue
        for index, (correct, score) in enumerate(zip(DATA[method][kind]['correct'], DATA[method][kind]['scores']), 1):
            rows.append(dict(id=('N' if kind == 'normal' else 'C') + str(index),
                             group=kind, correct=correct, score=score))
    return rows


def inspect_case(method, case_id):
    for row in records(method):
        if row['id'] == case_id:
            return row
    raise ValueError('未知のケースです')


def aggregate(method, threshold=0, group='all'):
    integer(threshold, 'threshold')
    rows = records(method, group)
    selected = [r for r in rows if r['score'] >= threshold]
    total, answered = len(rows), len(selected)
    correct = sum(r['correct'] for r in selected)
    return dict(total=total, correct=sum(r['correct'] for r in rows),
                answered=answered, correct_answered=correct, withheld=total-answered,
                accuracy=correct/answered if answered else None,
                coverage=answered/total, correct_per_total=correct/total,
                selected_ids=[r['id'] for r in selected])


def choose(candidates, min_accuracy, min_coverage):
    """固定候補の条件照合。丸め前の整数積で境界を比較する。"""
    integer(min_accuracy, 'min_accuracy')
    integer(min_coverage, 'min_coverage')
    eligible = [deepcopy(c) for c in candidates if c['answered'] > 0
                and c['correct_answered']*100 >= min_accuracy*c['answered']
                and c['answered']*100 >= min_coverage*c['total']]
    eligible.sort(key=lambda c: (c['cost'], -c['coverage'], -c['accuracy'], c['method'], c['threshold']))
    return eligible[0] if eligible else None


def evaluate(config=None):
    c = configuration(config)
    groups, mixture, answers, candidates = {}, {}, {}, []
    for method in ('A', 'B'):
        groups[method] = {kind: aggregate(method, 0, kind) for kind in ('all', 'normal', 'changed')}
        normal, changed = groups[method]['normal'], groups[method]['changed']
        mixture[method] = (normal['correct']*c['normal_percent'] + changed['correct']*(100-c['normal_percent']))/600
        cost = c['cost_a' if method == 'A' else 'cost_b']
        answers[method] = aggregate(method, c['threshold'])
        answers[method]['cost'] = 12*cost + answers[method]['withheld']*c['review_cost']
        for threshold in (0, 80, 100):
            row = dict(method=method, threshold=threshold, **aggregate(method, threshold))
            row['cost'] = 12*cost + row['withheld']*c['review_cost']
            row['eligible'] = (row['answered'] > 0 and row['correct_answered']*100 >= c['min_accuracy']*row['answered']
                               and row['answered']*100 >= c['min_coverage']*12)
            candidates.append(row)
    return dict(config=c, groups=groups, mixture=mixture, answers=answers,
                candidates=candidates, best=choose(candidates, c['min_accuracy'], c['min_coverage']))


def experiment_configs():
    cases = {}
    for name, field, value, base_threshold in (
        ('threshold', 'threshold', 80, 0), ('mixture', 'normal_percent', 75, 0),
        ('review', 'review_cost', 10, 80), ('cost_b', 'cost_b', 1, 80),
        ('coverage', 'min_coverage', 40, 0), ('accuracy', 'min_accuracy', 90, 0)):
        base = dict(BASE, threshold=base_threshold)
        changed = dict(base, **{field: value})
        cases[name] = dict(base=base, changed=changed)
    return cases


def rounded(value):
    if isinstance(value, dict):
        return {k: rounded(v) for k, v in value.items()}
    if isinstance(value, list):
        return [rounded(v) for v in value]
    return round(value, 6) if isinstance(value, float) else value


def demo():
    return rounded(dict(baseline=evaluate(), changes={name: dict(before=evaluate(case['base']), after=evaluate(case['changed']))
                                                    for name, case in experiment_configs().items()}))




## 基準結果
A 7/12・B 8/12。候補選択はA・しきい値80・費用32。表示値だけで安全性は判断できません。

In [ ]:
baseline_config = deepcopy(BASE)
baseline_data = deepcopy(DATA)
baseline = evaluate(baseline_config)
baseline_result = demo()
print(json.dumps(rounded(baseline), ensure_ascii=False, indent=2))


## 操作テーマ1：ケースを確認

問い・予想：操作教材と同じ結果になるでしょうか。実行前に予想を書きます。

手順：次のセルを実行。想定結果：N6はスコア90で不正解。

説明：変えた条件・分母・限界を挙げてください。復帰：再実行で同じ結果に戻ります。これは基準入力を編集しません。

In [ ]:
observation_case = inspect_case('A', 'N6')
print(observation_case)


## 操作テーマ2：条件別に集計

問い・予想：操作教材と同じ結果になるでしょうか。実行前に予想を書きます。

手順：次のセルを実行。想定結果：通常条件A5/6・B4/6。

説明：変えた条件・分母・限界を挙げてください。復帰：再実行で同じ結果に戻ります。これは基準入力を編集しません。

In [ ]:
observation_groups = {m: aggregate(m, 0, 'normal') for m in ('A', 'B')}
print(observation_groups)


## 操作テーマ3：比率を仮定

問い・予想：操作教材と同じ結果になるでしょうか。実行前に予想を書きます。

手順：次のセルを実行。想定結果：推定A17/24・B2/3。

説明：変えた条件・分母・限界を挙げてください。復帰：再実行で同じ結果に戻ります。これは基準入力を編集しません。

In [ ]:
observation_mixture = evaluate({'normal_percent': 75})
print(observation_mixture['mixture'])


## 操作テーマ4：回答と保留

問い・予想：操作教材と同じ結果になるでしょうか。実行前に予想を書きます。

手順：次のセルを実行。想定結果：A6/7・回答率7/12、B5/5・回答率5/12。

説明：変えた条件・分母・限界を挙げてください。復帰：再実行で同じ結果に戻ります。これは基準入力を編集しません。

In [ ]:
observation_abstain = evaluate({'threshold': 80})
print(observation_abstain['answers'])


## 操作テーマ5：確認費用を計算

問い・予想：操作教材と同じ結果になるでしょうか。実行前に予想を書きます。

手順：次のセルを実行。想定結果：しきい値80の比較基準から確認費用だけ変更し、62・106。

説明：変えた条件・分母・限界を挙げてください。復帰：再実行で同じ結果に戻ります。これは基準入力を編集しません。

In [ ]:
observation_costs = evaluate({'threshold': 80, 'review_cost': 10})
print({m: observation_costs['answers'][m]['cost'] for m in ('A', 'B')})


## 操作テーマ6：要件を照合

問い・予想：操作教材と同じ結果になるでしょうか。実行前に予想を書きます。

手順：次のセルを実行。想定結果：必要回答率50%も満たす候補はなくNone。

説明：変えた条件・分母・限界を挙げてください。復帰：再実行で同じ結果に戻ります。これは基準入力を編集しません。

In [ ]:
observation_choice = evaluate({'min_accuracy': 90})
print(observation_choice['best'])


## 変更1：しきい値0→80

問い・予想：何が変わり、何が変わらないか、実行前に手元へ書きましょう。

手順：この例の基準設定をコピーして一項目だけ変更します。想定結果：Aは6/7・回答率7/12、Bは5/5・回答率5/12。

説明：分母・変えた条件・測っていない事項を挙げてください。復帰：このセルを再実行。全体はカーネル再起動後の全セル実行。

In [ ]:
trial_threshold_base = deepcopy(experiment_configs()['threshold']['base'])
trial_threshold_config = deepcopy(trial_threshold_base)
trial_threshold_config.update({k: v for k, v in experiment_configs()['threshold']['changed'].items() if v != trial_threshold_base[k]})
trial_threshold_before = evaluate(trial_threshold_base)
trial_threshold = evaluate(trial_threshold_config)
print(json.dumps(rounded(trial_threshold), ensure_ascii=False, indent=2))


## 変更2：通常条件50→75%

問い・予想：何が変わり、何が変わらないか、実行前に手元へ書きましょう。

手順：この例の基準設定をコピーして一項目だけ変更します。想定結果：推定A17/24・B2/3。元の固定結果は変わらない。

説明：分母・変えた条件・測っていない事項を挙げてください。復帰：このセルを再実行。全体はカーネル再起動後の全セル実行。

In [ ]:
trial_mixture_base = deepcopy(experiment_configs()['mixture']['base'])
trial_mixture_config = deepcopy(trial_mixture_base)
trial_mixture_config.update({k: v for k, v in experiment_configs()['mixture']['changed'].items() if v != trial_mixture_base[k]})
trial_mixture_before = evaluate(trial_mixture_base)
trial_mixture = evaluate(trial_mixture_config)
print(json.dumps(rounded(trial_mixture), ensure_ascii=False, indent=2))


## 変更3：確認費用4→10（基準しきい値80）

問い・予想：何が変わり、何が変わらないか、実行前に手元へ書きましょう。

手順：この例の基準設定をコピーして一項目だけ変更します。想定結果：総費用A62・B106。

説明：分母・変えた条件・測っていない事項を挙げてください。復帰：このセルを再実行。全体はカーネル再起動後の全セル実行。

In [ ]:
trial_review_base = deepcopy(experiment_configs()['review']['base'])
trial_review_config = deepcopy(trial_review_base)
trial_review_config.update({k: v for k, v in experiment_configs()['review']['changed'].items() if v != trial_review_base[k]})
trial_review_before = evaluate(trial_review_base)
trial_review = evaluate(trial_review_config)
print(json.dumps(rounded(trial_review), ensure_ascii=False, indent=2))


## 変更4：B単価3→1（基準しきい値80）

問い・予想：何が変わり、何が変わらないか、実行前に手元へ書きましょう。

手順：この例の基準設定をコピーして一項目だけ変更します。想定結果：B総費用40。A32は変わらない。

説明：分母・変えた条件・測っていない事項を挙げてください。復帰：このセルを再実行。全体はカーネル再起動後の全セル実行。

In [ ]:
trial_cost_b_base = deepcopy(experiment_configs()['cost_b']['base'])
trial_cost_b_config = deepcopy(trial_cost_b_base)
trial_cost_b_config.update({k: v for k, v in experiment_configs()['cost_b']['changed'].items() if v != trial_cost_b_base[k]})
trial_cost_b_before = evaluate(trial_cost_b_base)
trial_cost_b = evaluate(trial_cost_b_config)
print(json.dumps(rounded(trial_cost_b), ensure_ascii=False, indent=2))


## 変更5：必要回答率50→40%

問い・予想：何が変わり、何が変わらないか、実行前に手元へ書きましょう。

手順：この例の基準設定をコピーして一項目だけ変更します。想定結果：適合候補はA80・B80。選択はA80のまま。

説明：分母・変えた条件・測っていない事項を挙げてください。復帰：このセルを再実行。全体はカーネル再起動後の全セル実行。

In [ ]:
trial_coverage_base = deepcopy(experiment_configs()['coverage']['base'])
trial_coverage_config = deepcopy(trial_coverage_base)
trial_coverage_config.update({k: v for k, v in experiment_configs()['coverage']['changed'].items() if v != trial_coverage_base[k]})
trial_coverage_before = evaluate(trial_coverage_base)
trial_coverage = evaluate(trial_coverage_config)
print(json.dumps(rounded(trial_coverage), ensure_ascii=False, indent=2))


## 変更6：必要正答率80→90%

問い・予想：何が変わり、何が変わらないか、実行前に手元へ書きましょう。

手順：この例の基準設定をコピーして一項目だけ変更します。想定結果：必要回答率50%も満たす候補はない。

説明：分母・変えた条件・測っていない事項を挙げてください。復帰：このセルを再実行。全体はカーネル再起動後の全セル実行。

In [ ]:
trial_accuracy_base = deepcopy(experiment_configs()['accuracy']['base'])
trial_accuracy_config = deepcopy(trial_accuracy_base)
trial_accuracy_config.update({k: v for k, v in experiment_configs()['accuracy']['changed'].items() if v != trial_accuracy_base[k]})
trial_accuracy_before = evaluate(trial_accuracy_base)
trial_accuracy = evaluate(trial_accuracy_config)
print(json.dumps(rounded(trial_accuracy), ensure_ascii=False, indent=2))


## 基準の保持とエラー対処

空欄や文字列ではなく0〜100の整数を指定します。`ValueError`は値・キーを見直します。`NameError`は上の定義セルが未実行です。カーネルを選べない場合は同梱guide.htmlの手順を確認します。全件保留のNoneは正常です。

問い：同じ固定データを再集計したことは、新しい独立評価を行ったことになるでしょうか。

In [ ]:
assert baseline_config == BASE
assert baseline_data == DATA
assert baseline == evaluate(BASE)
assert baseline_result == demo()
print('基準入力・結果を保持しました')
